# C4. QPU budget and form numbers

Turns the pilot's measured cost into the hour estimate for the Credits form, and counts two-qubit gates after
transpilation. Writes `credits/outputs/budget_for_form.md` with the answers ready to paste.

If `pilot_summary.json` is missing (no hardware pilot yet), it uses IBM's quick formula and says so.

In [1]:
BLOCKS = [   # the plan in the application. Edit here and rerun.
    dict(block="A. Scaling", tests="20, 40, 60, 80, 100, 127, 156 qubits at 1 and 2 layers", configs=14, tumors=1000, shots=1024),
    dict(block="B. Deep and strong regime", tests="156 qubits at 4, 5, 6 layers; 2 layers at bandwidth 0.5 and 1.0", configs=5, tumors=1000, shots=1024),
    dict(block="C. Shot noise", tests="156 qubits at 1 and 2 layers", configs=2, tumors=300, shots=4096),
    dict(block="D. NAS feature map", tests="20, 100, 156 qubits", configs=3, tumors=1000, shots=1024),
    dict(block="E. Reproducibility", tests="156 qubits, 1 layer, second Heron device", configs=1, tumors=1000, shots=1024),
]
TUMORS_PER_JOB = 100     # bigger jobs mean fewer per-job overheads
CONTINGENCY    = 0.12    # failed jobs, recalibration, mitigation overhead
TARGET_HOURS   = 8.5     # what the application asks for
DEFAULT_US_PER_SHOT, DEFAULT_OVERHEAD_S = 350.0, 2.0      # IBM's quick formula

In [2]:
import os, sys, json, time, math, warnings
from pathlib import Path
# Find the project folder (the one holding the pqk100 package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd(), Path(r"D:\Active Research Projects\PQK100")]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "pqk100" / "__init__.py").exists() and (p / "credits").is_dir():
                return p.resolve()
    raise RuntimeError(f"Cannot find the PQK100 project folder from {Path.cwd()}. "
                       "Set ROOT = Path(r'D:\\Active Research Projects\\PQK100') by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project folder:", ROOT)
OUT = ROOT / "credits" / "outputs"; OUT.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
pd.set_option("display.width", 200)

from pqk100.backends import local_backend
from pqk100.featuremap import FEATURE_MAPS, cone_stats
from pqk100.hardware import build, circuit_report
from pqk100.lattice import patch

project folder: D:\Active Research Projects\PQK100


## C4.1 Rate

In [3]:
f = OUT / "pilot_summary.json"
pilot = json.loads(f.read_text()) if f.exists() else None
if pilot and pilot.get("mode") == "hardware" and pilot.get("us_per_shot"):
    US, OV, SRC = pilot["us_per_shot"], pilot["overhead_s_per_job"] or 0.0, f"measured on {pilot['backend']}, {pilot['date']}"
else:
    US, OV, SRC = DEFAULT_US_PER_SHOT, DEFAULT_OVERHEAD_S, "IBM quick formula (no hardware pilot yet)"
print(f"rate: {US:.0f} us per shot + {OV:.1f} s per job   [{SRC}]")

rate: 284 us per shot + 11.4 s per job   [measured on ibm_fez, 2026-10-18]


## C4.2 Hours per block

In [4]:
def block_hours(b, us=US, ov=OV):
    jobs = math.ceil(b["tumors"] / TUMORS_PER_JOB)
    return b["configs"] * (jobs * ov + b["tumors"] * 3 * b["shots"] * us * 1e-6) / 3600

tab = pd.DataFrame(BLOCKS)
tab["QPU hours"] = [block_hours(b) for b in BLOCKS]
sub = tab["QPU hours"].sum()
total = sub * (1 + CONTINGENCY)
ask = math.ceil(total * 2) / 2                       # round up to half an hour
tab.loc[len(tab)] = ["Contingency", f"{CONTINGENCY:.0%} of the blocks", None, None, None, total - sub]
tab.loc[len(tab)] = ["Total", "", tab.configs.sum(), None, None, total]
print(f"Estimate for the form: {ask:g} hours  (computed {total:.2f})")
tab.round(2)

Estimate for the form: 8 hours  (computed 7.74)


,block,tests,configs,tumors,shots,QPU hours
0,A. Scaling,"20, 40, 60, 80, 100, 127, 156 qubits at 1 and ...",14,1000,1024,3.84
1,B. Deep and strong regime,"156 qubits at 4, 5, 6 layers; 2 layers at band...",5,1000,1024,1.37
2,C. Shot noise,156 qubits at 1 and 2 layers,2,300,4096,0.60
3,D. NAS feature map,"20, 100, 156 qubits",3,1000,1024,0.82
4,E. Reproducibility,"156 qubits, 1 layer, second Heron device",1,1000,1024,0.27
5,Contingency,12% of the blocks,None,None,None,0.83
6,Total,,25,None,None,7.74


## C4.3 Room for more shots

If the measured rate is lower than 0.35 ms, the same request buys more shots. This finds the most shots Block A
can use while the total stays within `TARGET_HOURS`.

In [5]:
best = None
for shots in (1024, 2048, 4096, 8192):
    trial = [dict(b, shots=shots) if b["block"].startswith("A.") else b for b in BLOCKS]
    tot = sum(block_hours(b) for b in trial) * (1 + CONTINGENCY)
    print(f"Block A at {shots:5d} shots: total {tot:5.2f} h")
    if tot <= TARGET_HOURS:
        best = shots
print(f"\nWithin {TARGET_HOURS} h, Block A can use {best} shots." if best else f"\nEven 1,024 shots exceed {TARGET_HOURS} h: trim the plan or raise the request.")

Block A at  1024 shots: total  7.74 h
Block A at  2048 shots: total 11.54 h
Block A at  4096 shots: total 19.14 h
Block A at  8192 shots: total 34.35 h

Within 8.5 h, Block A can use 1024 shots.


## C4.4 Two-qubit gates after transpilation

On the FakeFez lattice, 156 qubits. Each RZZ becomes two CZ gates, so a layer costs two CZ per coupler.

In [6]:
backend = local_backend()
phys, edges = patch(backend, 156)
rows = []
for reps in (1, 2, 3, 4, 5, 6):
    ops = FEATURE_MAPS["zz"](156, edges, reps=reps)
    isa, _, _ = build(156, ops, phys, backend)
    r = circuit_report(isa, backend)
    size, _, _ = cone_stats(156, ops)
    rows.append({"layers": reps, "CZ gates": r["twoq_gates"], "CZ depth": r["twoq_depth"],
                 "largest light cone (qubits)": int(size.max()), "circuit (us)": round(r["circuit_s"] * 1e6, 1)})
gates = pd.DataFrame(rows).set_index("layers")
print(f"{len(edges)} couplers in the 156-qubit patch")
gates

176 couplers in the 156-qubit patch


,CZ gates,CZ depth,largest light cone (qubits),circuit (us)
layers,,,,
1,352,6,4,0.8
2,704,12,7,1.5
3,1056,18,13,2.3
4,1408,24,19,3.1
5,1760,30,31,3.8
6,2112,36,41,4.6


## C4.5 Form answers

In [7]:
cz1, cz2, cz6 = gates.loc[1, "CZ gates"], gates.loc[2, "CZ gates"], gates.loc[6, "CZ gates"]
if pilot and pilot.get("mode") == "hardware":
    pilot_line = (f"Yes. Pilot on {pilot['backend']}: {pilot['qubits']} qubits, {pilot['us_per_shot']:.0f} \u00b5s billed per shot, "
                  f"{100 * pilot['signal_retained_mean']:.0f}% of the exact signal retained on average across qubits.")
    qubits_now = f"156 in simulation on the ibm_fez lattice; {pilot['qubits']} on ibm_fez hardware (pilot)"
else:
    pilot_line = "[run notebook C3 on hardware first]"
    qubits_now = "156 in simulation on the ibm_fez lattice; [qubits] on ibm_fez hardware (pilot)"

lines = [
    f"# Credits form numbers ({time.strftime('%Y-%m-%d')})", "",
    f"Rate used: {US:.0f} us per shot + {OV:.1f} s per job ({SRC}).", "",
    "| Block | What it tests | Configurations | Tumors | Shots | QPU hours |", "| --- | --- | --- | --- | --- | --- |",
]
for _, b in tab.iterrows():
    cells = [b.block, b.tests, "" if pd.isna(b.configs) else int(b.configs), "" if pd.isna(b.tumors) else f"{int(b.tumors):,}",
             "" if pd.isna(b.shots) else f"{int(b.shots):,}", f"{b['QPU hours']:.1f}"]
    lines.append("| " + " | ".join(map(str, cells)) + " |")
lines += [
    "", "## Form fields", "",
    f"- Were you able to determine feasibility via the Open Plan: {pilot_line}",
    f"- Number of physical qubits to date: {qubits_now}",
    "- Target number of physical qubits: 156",
    f"- Number of 2-qubit gates (after transpilation): {cz1} CZ per layer at 156 qubits. Main study: {cz1} to {cz2}. Deep stress test: up to {cz6:,}.",
    f"- Estimate the quantum compute time: {ask:g} hours",
]
text = "\n".join(lines) + "\n"
(OUT / "budget_for_form.md").write_text(text, encoding="utf-8")
print(text)

# Credits form numbers (2026-10-18)

Rate used: 284 us per shot + 11.4 s per job (measured on ibm_fez, 2026-10-18).

| Block | What it tests | Configurations | Tumors | Shots | QPU hours |
| --- | --- | --- | --- | --- | --- |
| A. Scaling | 20, 40, 60, 80, 100, 127, 156 qubits at 1 and 2 layers | 14 | 1,000 | 1,024 | 3.8 |
| B. Deep and strong regime | 156 qubits at 4, 5, 6 layers; 2 layers at bandwidth 0.5 and 1.0 | 5 | 1,000 | 1,024 | 1.4 |
| C. Shot noise | 156 qubits at 1 and 2 layers | 2 | 300 | 4,096 | 0.6 |
| D. NAS feature map | 20, 100, 156 qubits | 3 | 1,000 | 1,024 | 0.8 |
| E. Reproducibility | 156 qubits, 1 layer, second Heron device | 1 | 1,000 | 1,024 | 0.3 |
| Contingency | 12% of the blocks |  |  |  | 0.8 |
| Total |  | 25 |  |  | 7.7 |

## Form fields

- Were you able to determine feasibility via the Open Plan: Yes. Pilot on ibm_fez: 156 qubits, 284 µs billed per shot, 89% of the exact signal retained on average across qubits.
- Number of physical qubits to date: 156